# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishnu-varthan1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

The baseline prioritizes content pages that have meaningful search visibility but may need attention because of weak click-through performance or outdated content.

The rule will use observable metrics such as impressions, CTR, average search position, and days since last update.

Each selected page will receive one reason code and one recommended action. The score will be transparent and based on fixed thresholds rather than trained model weights.

The ranking is intended for decision support, not as proof that updating a page will improve its performance.

In [2]:
!git clone https://github.com/vishnu-varthan1/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 135 (delta 47), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.86 MiB | 14.30 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print("Checking dataset...")
print("Dataset exists:", DATA_PATH.exists())

Checking dataset...
Dataset exists: True


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
display(df.head())

Shape: (30000, 44)
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
print("Dataset shape:", df.shape)

print("\nAvailable columns:")
for col in df.columns:
    print(col)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDataset preview:")
display(df.head(5))

Dataset shape: (30000, 44)

Available columns:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct

Missing values:


,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468



Dataset preview:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
print("DATASET SHAPE:", df.shape)

print("\nALL COLUMNS:")
print(df.columns.tolist())

print("\nDATA TYPES:")
display(df.dtypes)

print("\nSIGNAL SUMMARY:")
display(
    df[[
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "search_volume",
        "trend_pct"
    ]].describe()
)

DATASET SHAPE: (30000, 44)

ALL COLUMNS:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

DATA TYPES:


,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64



SIGNAL SUMMARY:


,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,search_volume,trend_pct
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000,27532.000000,26612.000000
mean,46.098300,5200.366300,16.097333,0.510733,16.34238,158.882391,-4.785969
std,42.078709,16838.019547,75.076958,3.279162,15.21679,1518.270825,473.861780
min,1.000000,1.000000,0.000000,0.000000,0.00000,0.000000,-100.000000
25%,20.000000,81.000000,0.000000,0.000000,6.20000,0.000000,-62.600000
50%,20.000000,731.000000,1.000000,0.070000,10.80000,10.000000,-33.500000
75%,104.000000,3615.250000,7.000000,0.290000,22.30000,20.000000,0.000000
max,373.000000,517715.000000,4178.000000,100.000000,245.00000,74000.000000,44900.000000


In [9]:
# ML-07: Signal audit

# Signal 1: CTR vs position
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-0.001, 3, 10, 20, 50, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21-50", "50+"]
)

df.loc[df["avg_position"] == 0, "position_bucket"] = "No position data"

position_audit = df.groupby(
    "position_bucket", observed=False
).agg(
    n=("ctr", "size"),
    median_ctr=("ctr", "median"),
    median_impressions=("impressions_90d", "median")
)

# Signal 2: Staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_audit = df.groupby(
    "staleness_bucket", observed=False
).agg(
    n=("content_id", "size"),
    median_impressions=("impressions_90d", "median"),
    median_clicks=("clicks_90d", "median"),
    median_ctr=("ctr", "median")
)

print("SIGNAL 1 — CTR vs POSITION")
display(position_audit)

print("SIGNAL 2 — CONTENT STALENESS")
display(staleness_audit)

TypeError: Cannot setitem on a Categorical with a new category (No position data), set the categories first

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# ML-07: Weak picks and leakage check

# 1. Check whether forbidden columns are used in scoring
forbidden_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win"
]

# Define the features permitted in the baseline
allowed_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

used_features = allowed_features.copy()

leakage_check = [
    col for col in used_features
    if col in forbidden_columns
]

print("LEAKAGE CHECK")
print("Forbidden features used:", leakage_check)
print("Result:", "PASS" if not leakage_check else "FAIL")

# 2. Inspect potentially weak records
weak_picks = df.head(20).copy()

print("\nPOTENTIAL WEAK PICKS")
print("Low-impression records:")

display(
    weak_picks[
        weak_picks["impressions_90d"] < 100
    ][[
        "content_id",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]]
)

print("\nRecords without position data:")

display(
    weak_picks[
        weak_picks["avg_position"] == 0
    ][[
        "content_id",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]]
)

LEAKAGE CHECK
Forbidden features used: []
Result: PASS

POTENTIAL WEAK PICKS
Low-impression records:


,content_id,impressions_90d,ctr,avg_position
6,content_9a34b442b552,20,0.00,7.0
11,content_5a3e876cf7f7,1,0.00,0.0
15,content_689414059706,38,23.68,7.8
19,content_af865035b328,99,2.02,6.9



Records without position data:


,content_id,impressions_90d,ctr,avg_position
11,content_5a3e876cf7f7,1,0.0,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.